# 01. Exploratory Data Analysis: NASA FIRMS Active Fire Archives

> **Notice:** This notebook is for **exploration only**. All production data pipelines, feature engineering, modeling, and serving logic are maintained under `src/`.

In [ ]:
import sys
from pathlib import Path

# Add project root to path for src imports
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_FIRMS_DIR, GRID_SIZE_DEG
from src.data.grid import coords_to_grid_id, generate_grid_mesh
from src.data.kbdi import compute_kbdi_timeseries, classify_kbdi_danger

sns.set_theme(style="whitegrid")
print("Project Root:", ROOT_DIR)

## 1. Inspect Raw NASA FIRMS CSV Data

In [ ]:
firms_files = list(RAW_FIRMS_DIR.glob("*.csv"))
print("FIRMS CSV Files found:", [f.name for f in firms_files])

if firms_files:
    sample_df = pd.read_csv(firms_files[0], nrows=1000)
    display(sample_df.head())
    print("\nDataset Columns:", list(sample_df.columns))
    print("\nSummary Statistics:")
    display(sample_df[['brightness', 'scan', 'track', 'confidence', 'frp']].describe())

## 2. Spatial Distribution & Grid Discretization

In [ ]:
if firms_files:
    sample_df['grid_id'] = coords_to_grid_id(sample_df['latitude'], sample_df['longitude'], grid_size=GRID_SIZE_DEG)
    print("Unique grid cells in sample:", sample_df['grid_id'].nunique())
    print("Top active cells in sample:")
    display(sample_df['grid_id'].value_counts().head(10))
    
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        sample_df['longitude'],
        sample_df['latitude'],
        c=sample_df['frp'],
        cmap='YlOrRd',
        alpha=0.6,
        s=20
    )
    plt.colorbar(scatter, label='Fire Radiative Power (MW)')
    plt.title('Spatial Distribution of Hotspots (FRP Intensity)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

## 3. Keetch-Byram Drought Index (KBDI) Demonstration

In [ ]:
# Simulate 90-day dry spell followed by monsoon rainfall
dates = pd.date_range("2023-03-01", periods=90, freq="D")
temps = 28 + 12 * np.sin(np.linspace(0, np.pi, 90)) + np.random.normal(0, 1.5, 90)
rains = np.zeros(90)
rains[70:75] = [25.0, 40.0, 15.0, 8.0, 30.0]  # Monsoon storm onset

sim_weather = pd.DataFrame({"date": dates, "temp_max_c": temps, "precip_24h_mm": rains})
sim_weather["kbdi"] = compute_kbdi_timeseries(sim_weather, initial_kbdi=250.0)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(sim_weather['date'], sim_weather['kbdi'], color='red', label='KBDI (Drought Index)')
ax1.axhline(600, color='darkred', linestyle='--', alpha=0.5, label='Extreme Hazard Threshold (600)')
ax1.axhline(400, color='orange', linestyle='--', alpha=0.5, label='High Hazard Threshold (400)')
ax1.set_ylabel('KBDI (0 - 800)', color='red')
ax1.set_ylim(0, 800)

ax2 = ax1.twinx()
ax2.bar(sim_weather['date'], sim_weather['precip_24h_mm'], color='blue', alpha=0.3, width=1.0, label='Rainfall (mm)')
ax2.set_ylabel('24h Rainfall (mm)', color='blue')

plt.title('KBDI Accumulation During Dry Spell & Plunge After Rainfall')
plt.show()

## 4. Next Steps
- Production feature extraction is handled in `src.features.build_features`.
- Temporal model training with probability calibration is run via `src.models.train`.
- Live serving and dashboard are located in `src.api.main` and `src.dashboard.app`.